<a href="https://colab.research.google.com/github/enedenao/tp_labo/blob/Jose/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP Grupal — Laboratorio de Métodos Cuantitativos (FCE UBA, 2C 2026)

**Integrantes:** Bautista Cortés · Gabriel Goldadler · José Luis Joannás · Naomi Asato · @dagut-rrd

**Repositorio:** https://github.com/enedenao/tp_labo

**Pregunta de investigación (versión actualizada):**

¿Qué personajes concentran el diálogo en Los Simpson, y existe una asociación entre el protagonismo de los personajes principales y la valoración de los episodios en IMDb que se mantenga a lo largo de las décadas?

## 0. Configuración del entorno

Importamos las bibliotecas que vamos a usar en todo el análisis:

- **pandas**: carga y manipulación del dataset (DataFrames, `groupby`, filtros).
- **numpy**: operaciones numéricas auxiliares.
- **matplotlib** y **seaborn**: gráficos. Seaborn se apoya en matplotlib y simplifica gráficos estadísticos (boxplots, barras agrupadas).
- **os**: crear la carpeta donde se guardan los gráficos exportados para el informe.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("figuras", exist_ok=True)

> **Asistencia de IA:** Claude (Claude Project "tp labo"). Propuso el uso de os

## 1. Carga del dataset

**Dataset:** The Simpsons Data recopilación de Todd Schneider

**Fuente:** publicado en Kaggle

**Página del recurso:** https://github.com/sujanjoejacob/Text-mining-with-Simpsons-Data


Cargamos el CSV/ZIP directamente desde el repo, para que el notebook sea reproducible sin depender de archivos locales.

In [2]:
URL_EPISODIOS = "https://raw.githubusercontent.com/enedenao/tp_labo/main/data/simpsons_episodes.csv"
URL_DIALOGOS = "https://raw.githubusercontent.com/enedenao/tp_labo/main/data/simpsons_script_lines.csv.zip"

episodios = pd.read_csv(URL_EPISODIOS)
dialogos = pd.read_csv(URL_DIALOGOS, on_bad_lines="skip", low_memory=False)

print(f"Episodios: {episodios.shape[0]:,} filas | {episodios.shape[1]} columnas")
print(f"Dialogos:   {dialogos.shape[0]:,} filas | {dialogos.shape[1]} columnas")

Episodios: 600 filas | 13 columnas
Dialogos:   158,248 filas | 13 columnas


**Decisión de limpieza en la carga:** algunas filas del archivo de diálogos están mal armadas: tienen comas de más dentro del texto, y pandas no puede separar bien las columnas. Por defecto, eso hace que la carga falle. Con `on_bad_lines="skip"` se descartan esas filas: son 23 sobre 158.271 (menos del 0,02%), por lo que no afectan el análisis.

> **Asistencia de IA:** Claude (Claude Project "tp labo"). Se le compartió el CSV de diálogos y detectó las filas mal formadas; propuso `on_bad_lines="skip"`.

## 3.1 Exploración inicial

### a) Primeras y últimas filas

Mostramos el inicio y el final de cada tabla para ver cómo vienen los datos y detectar problemas a simple vista.

In [3]:
episodios.head()

,id,title,original_air_date,production_code,season,number_in_season,number_in_series,us_viewers_in_millions,views,imdb_rating,imdb_votes,image_url,video_url
0,10,Homer's Night Out,1990-03-25,7G10,1,10,10,30.3,50816.0,7.4,1511.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/275197507879
1,12,Krusty Gets Busted,1990-04-29,7G12,1,12,12,30.4,62561.0,8.3,1716.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/288019523914
2,14,"Bart Gets an ""F""",1990-10-11,7F03,2,1,14,33.6,59575.0,8.2,1638.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/260539459671
3,17,Two Cars in Every Garage and Three Eyes on Eve...,1990-11-01,7F01,2,4,17,26.1,64959.0,8.1,1457.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/260537411822
4,19,Dead Putting Society,1990-11-15,7F08,2,6,19,25.4,50691.0,8.0,1366.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/260539459670


In [4]:
episodios.tail()

,id,title,original_air_date,production_code,season,number_in_season,number_in_series,us_viewers_in_millions,views,imdb_rating,imdb_votes,image_url,video_url
595,500,At Long Last Leave,2012-02-19,PABF07,23,14,500,5.77,45226.0,7.0,580.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/230541891832
596,502,How I Wet Your Mother,2012-03-11,PABF08,23,16,502,4.97,44818.0,7.3,541.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/230597187582
597,511,Adventures in Baby-Getting,2012-11-04,PABF18,24,3,511,5.65,39687.0,6.9,460.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/221688899646
598,542,Diggs,2014-03-09,SABF08,25,12,542,2.69,39292.0,6.4,473.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/310529091639
599,548,Days of Future Future,2014-04-13,SABF13,25,18,548,3.64,55742.0,7.0,504.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/310257731892


In [5]:
dialogos.head()

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,true,464,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31.0
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,true,9,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3.0
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,true,464,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22.0
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,true,9,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5.0
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,true,40,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33.0


In [6]:
dialogos.tail()

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
158243,9544,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,true,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2.0
158244,9545,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,true,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10.0
158245,9546,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,true,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1.0
158246,9547,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,true,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6.0
158247,9548,32,208,"JANEY: No, that means she was faking it.",845000,true,4,3.0,JANEY,Springfield Elementary School,"No, that means she was faking it.",no that means she was faking it,7.0


### b) Estructura: dimensiones, columnas, tipos de dato y cobertura

`.dtypes` muestra el tipo de dato de cada columna. Una columna que debería ser numérica pero aparece como texto (`object`) suele indicar datos sucios.

In [7]:
print(episodios.dtypes)

id                          int64
title                      object
original_air_date          object
production_code            object
season                      int64
number_in_season            int64
number_in_series            int64
us_viewers_in_millions    float64
views                     float64
imdb_rating               float64
imdb_votes                float64
image_url                  object
video_url                  object
dtype: object


In [8]:
print(dialogos.dtypes)

id                      int64
episode_id              int64
number                  int64
raw_text               object
timestamp_in_ms        object
speaking_line          object
character_id           object
location_id           float64
raw_character_text     object
raw_location_text      object
spoken_words           object
normalized_text        object
word_count            float64
dtype: object


In [9]:
print("Fecha de emisión: de", episodios["original_air_date"].min(), "a", episodios["original_air_date"].max())
print("Temporadas:", episodios["season"].nunique())
print("Episodios en la tabla de episodios:", episodios["id"].nunique())                    #nunique() cuenta cuántos valores distintos hay en una columna.
print("Episodios con líneas de diálogo:", dialogos["episode_id"].nunique())

Fecha de emisión: de 1989-12-17 a 2016-10-16
Temporadas: 28
Episodios en la tabla de episodios: 600
Episodios con líneas de diálogo: 564


Hay 36 episodios sin dialogo

### c) Calidad de los datos: nulos, duplicados y estadísticas descriptivas

Contamos valores faltantes y filas repetidas, y resumimos las columnas numéricas. Revisamos además los valores de `speaking_line`, que indica si una línea es diálogo o una acotación del guion.

In [10]:
print(episodios.isnull().sum()) #tenemos 3 episodios (imdb_rating) sin datos que representa el 0.5%
print("Filas duplicadas:", episodios.duplicated().sum()) #no hay duplicados

id                        0
title                     0
original_air_date         0
production_code           0
season                    0
number_in_season          0
number_in_series          0
us_viewers_in_millions    6
views                     4
imdb_rating               3
imdb_votes                3
image_url                 4
video_url                 4
dtype: int64
Filas duplicadas: 0


In [11]:
print(dialogos.isnull().sum()) #el exceso de nulos corresponden a acotaciones del guion
print("Filas duplicadas:", dialogos.duplicated().sum()) #no tenemos filas duplicadas

id                        0
episode_id                0
number                    0
raw_text                  0
timestamp_in_ms           0
speaking_line             0
character_id          17521
location_id             407
raw_character_text    17522
raw_location_text       408
spoken_words          26159
normalized_text       26184
word_count            26159
dtype: int64
Filas duplicadas: 0


In [12]:
episodios[["imdb_rating", "us_viewers_in_millions", "imdb_votes"]].describe()

,imdb_rating,us_viewers_in_millions,imdb_votes
count,597.000000,594.000000,597.000000
mean,7.386097,11.842795,832.365159
std,0.732439,6.414218,396.117937
min,4.500000,2.320000,104.000000
25%,6.900000,7.055000,560.000000
50%,7.300000,10.300000,697.000000
75%,8.000000,15.250000,1095.000000
max,9.200000,33.600000,3734.000000


**Rating**:

Promedio 7,4, mediana 7,3, desvío 0,73.

El 50% de los episodios está entre 6,9 y 8,0 (cuartiles 25% y 75%).

El mínimo, 4,5, está muy por debajo del resto: es un episodio atípico.

Que el rango sea estrecho importa para la pregunta: las diferencias de rating entre episodios son chicas, así que cualquier asociación con el protagonismo probablemente sea moderada.

**Audiencia**:

Promedio 11,8 millones, pero mediana 10,3. Cuando el promedio es mayor que la mediana, hay valores altos que “tiran” el promedio para arriba. El rango va de 2,3 a 33,6 millones, lo que sugiere una caída importante de la audiencia a lo largo del tiempo, a verificar en el análisis de gráfico.

In [13]:
dialogos["word_count"].describe()
#las 132.089 lineas con palabras registradas tienen en promedio casi 10 palabras, con una mediana de 8: predominan los dialogos breves

,word_count
count,132089.000000
mean,9.887175
std,8.055425
min,0.000000
25%,4.000000
50%,8.000000
75%,13.000000
max,122.000000


In [14]:
dialogos["speaking_line"].value_counts(dropna=False) #132.089 lineas habladas y 26.158 acotaciones
#hay una fila que es dialogo y quedo en columna equivocada

,count
speaking_line,
true,132089
false,26158
Guess what. I also play Frankenstein!,1


In [15]:
episodios["season"].value_counts().sort_index() #temporada 28 incompleta

,count
season,
1,13
2,22
3,24
4,22
5,22
6,25
7,25
8,25
9,25


### Hallazgos de la exploración

**Episodios**

- 600 episodios de 28 temporadas, emitidos entre el 17/12/1989 y el 16/10/2016. La mayoría de las temporadas tiene entre 20 y 25 episodios; la temporada 1 tiene 13, por ser más corta, y la **temporada 28 tiene solo 4**, (datos se recopilaron poco después del estreno de la 28°temporada).
- No hay filas duplicadas y los `id` van del 1 al 600 sin repetirse.
- Hay pocos nulos: 6 en audiencia (`us_viewers_in_millions`), 4 en `views` y 3 en `imdb_rating` e `imdb_votes`. Son menos del 1% de los episodios.
- `imdb_rating` va de 4,5 a 9,2, con promedio de 7,4 y desvío de 0,73: la mayoría de los episodios se concentra en un rango estrecho.
- La audiencia en EE.UU. va de 2,3 a 33,6 millones de espectadores: hay una diferencia muy grande entre los episodios más y menos vistos, que habrá que analizar en el tiempo.
- `original_air_date` está como texto (`object`), no como fecha, y no hay una columna de año. Para analizar por décadas hay que convertirla y crear esa columna.
- Las filas no están ordenadas cronológicamente (el `head()` arranca en el episodio 10 y el `tail()` termina en el 548).

**Diálogos**

- 158.248 líneas de guion (se descartaron 23 filas mal formadas al cargar), sin filas duplicadas.
- Solo 564 de los 600 episodios tienen líneas de diálogo: 36 episodios no tienen guion y no pueden incluirse en el análisis de protagonismo. [Completar: si se concentran en alguna época.]
- `speaking_line` no es booleana: viene como texto (`"true"` / `"false"`). Hay 132.089 líneas habladas y 26.158 acotaciones, más una fila corrupta con una frase del guion en esa columna.
- `timestamp_in_ms` y `character_id` deberían ser numéricas pero aparecen como texto (`object`), por esa misma fila corrupta.
- Los nulos en `spoken_words`, `normalized_text` y `word_count` (unos 26.000) corresponden a las acotaciones del guion, que no tienen palabras dichas. Los nulos en `character_id` y `raw_character_text` corresponden mayormente a acotaciones sin personaje.
- Las líneas con palabras tienen en promedio casi 10 palabras (mediana 8, máximo 122). Hay líneas con 0 palabras registradas.
- Hay 6.763 nombres de personajes distintos, y un mismo personaje puede aparecer con varios nombres: por ejemplo, 63 nombres contienen "Bart" ("Adult Bart", "Teenage Bart"), aunque "Bart Simpson" concentra la gran mayoría de las líneas. La búsqueda por texto incluye además personajes que no son Bart, como "Bartender", por lo que una unificación automática no sería confiable.

**Para la etapa de transformaciones:**

- Filtrar solo las líneas habladas (`speaking_line == "true"`), lo que también descarta la fila corrupta.
- Convertir `original_air_date` a fecha y crear las columnas de año y década.
- Convertir `timestamp_in_ms` y `character_id` a numéricas.
- Decidir cómo tratar los nulos de rating y audiencia, la temporada 28 (incompleta) y las variantes de nombres de personajes.

>**Asistencia de IA:** Claude (Claude Project "tp labo"). Se usó como apoyo para para revisar y mejorar todo lo trabajado.

## 3.2 Pregunta de investigación

**a) ¿Cuál es la pregunta?**

¿Qué personajes concentran el diálogo en Los Simpson, y existe una asociación entre el protagonismo de los personajes principales y la valoración de los episodios en IMDb que se mantenga a lo largo de las décadas?

Para que sea respondible con los datos disponibles, los conceptos se miden así:
- **Concentración del diálogo:** cantidad de palabras habladas por personaje (`character_id`) sobre el total de palabras habladas.
- **Personajes principales:** los cuatro con más palabras habladas, definidos a partir del ranking de los propios datos.
- **Protagonismo de un episodio:** porcentaje de las palabras habladas del episodio que dicen los personajes principales (`pct_principales`).
- **Valoración:** calificación del episodio en IMDb (`imdb_rating`).
- **Décadas:** tres períodos según la fecha de emisión (1989-1999, 2000-2009 y 2010-2016). No todos cubren diez años: la serie se estrenó a fines de 1989 y el dataset termina en 2016.
- **Asociación:** relación estadística entre protagonismo y valoración (por ejemplo, correlación y comparación entre períodos). No implica causalidad.

**b) ¿Por qué vale la pena responderla?**

La serie lleva más de 25 años en el aire y su elenco es muy amplio, con una familia central y cientos de personajes secundarios. Responder la pregunta ayuda a entender si los episodios en los que la familia principal concentra más diálogo son mejor o peor valorados por el público, y si esa relación cambió a lo largo de la historia de la serie. Ilumina una decisión típica de la producción de contenido: cuánto peso darle al elenco central frente a los personajes secundarios al escribir los guiones.

**c) ¿Qué respuesta encontraron?**

[Completar al finalizar el análisis, con los hallazgos concretos: quiénes concentran el diálogo, el sentido y la magnitud de la asociación con el rating IMDb, y si se mantiene entre períodos.]


## 3.3 Transformaciones

### Transformación 1: limpieza de los diálogos y construcción del protagonismo por episodio
**Qué se hace:**
1. Se conservan solo las líneas habladas (`speaking_line == "true"`), se convierten `timestamp_in_ms` y `character_id` a numéricas y se eliminan las líneas sin personaje.
2. Se arma el ranking de personajes por cantidad de palabras habladas y se definen como "principales" los cuatro primeros.
3. Se calcula, para cada episodio, el porcentaje de las palabras habladas que dicen los principales (`pct_principales`).

**Por qué:**
- El protagonismo se mide con lo que los personajes dicen, no con las acotaciones del guion (26.158 líneas). Al filtrar por `"true"` también se descarta la fila corrupta que hacía que `timestamp_in_ms` y `character_id` quedaran como texto.
- Se agrupa por `character_id` y no por el nombre escrito, porque un mismo personaje puede aparecer con variantes de nombre; el id es único. Definir los principales con el ranking evita elegirlos a mano y responde la primera parte de la pregunta.
- Se miden **palabras** y no líneas (criterio acordado por el grupo), porque una línea corta y una larga no pesan igual: las palabras reflejan mejor cuánto "habla" cada personaje.
- Se usa un **porcentaje** y no una cantidad para que episodios con más o menos diálogo sean comparables entre sí.


In [16]:
# 1. Líneas habladas (descarta acotaciones y la fila corrupta)
print("Antes:", dialogos.shape)
dialogos_h = dialogos[dialogos["speaking_line"] == "true"].copy()
print("Líneas habladas:", dialogos_h.shape)   # esperado: 132.089

# 2. Tipos numéricos y líneas sin personaje
dialogos_h["timestamp_in_ms"] = pd.to_numeric(dialogos_h["timestamp_in_ms"], errors="coerce")
dialogos_h["character_id"] = pd.to_numeric(dialogos_h["character_id"], errors="coerce")
n_antes = len(dialogos_h)
dialogos_h = dialogos_h.dropna(subset=["character_id"]).copy()
dialogos_h["character_id"] = dialogos_h["character_id"].astype(int)
print("Líneas sin personaje eliminadas:", n_antes - len(dialogos_h))

# 3. Ranking de personajes por palabras
ranking = dialogos_h.groupby("character_id").agg(
    nombre=("raw_character_text", "first"),
    palabras=("word_count", "sum"))
ranking = ranking.sort_values("palabras", ascending=False)
ranking["pct_del_total"] = ranking["palabras"] / ranking["palabras"].sum() * 100
display(ranking.head(10))
ids_principales = ranking.head(4).index.tolist()

# 4. Protagonismo por episodio (en palabras)
dialogos_h["es_principal"] = dialogos_h["character_id"].isin(ids_principales)
dialogos_h["palabras_principales"] = dialogos_h["word_count"] * dialogos_h["es_principal"]
por_ep = dialogos_h.groupby("episode_id").agg(
    n_palabras=("word_count", "sum"),
    palabras_principales=("palabras_principales", "sum"))
por_ep["pct_principales"] = por_ep["palabras_principales"] / por_ep["n_palabras"] * 100
por_ep = por_ep.reset_index()
print(por_ep["n_palabras"].describe())

Antes: (158248, 13)
Líneas habladas: (132089, 13)
Líneas sin personaje eliminadas: 2


,nombre,palabras,pct_del_total
character_id,,,
2,Homer Simpson,273780.0,20.963937
1,Marge Simpson,125666.0,9.622522
8,Bart Simpson,110814.0,8.485272
9,Lisa Simpson,100576.0,7.701326
15,C. Montgomery Burns,36623.0,2.804304
17,Moe Szyslak,32855.0,2.515780
3,Seymour Skinner,28138.0,2.154589
11,Ned Flanders,22909.0,1.754193
139,Krusty the Clown,20897.0,1.600129


count     564.000000
mean     2315.526596
std       267.219824
min       761.000000
25%      2150.750000
50%      2333.000000
75%      2493.250000
max      3066.000000
Name: n_palabras, dtype: float64


### Transformación 2: fecha y década, unión con el protagonismo y base final
**Qué se hace:**
1. Se convierte `original_air_date` a fecha y se crean `anio` y `decada` (1989-1999, 2000-2009 y 2010-2016).
2. Se une `episodios` con el protagonismo por episodio y se marca con `tiene_guion` qué episodios tienen diálogo.
3. La base final para el análisis conserva solo los episodios con guion.

**Por qué:**
- Sin variable temporal no se puede responder si la asociación se mantiene a lo largo de las décadas.
- **Aclaración sobre las décadas:** no todos los grupos cubren diez años. 1989-1999 incorpora 1989 porque la serie se estrenó el 17/12/1989 y un grupo de un solo episodio no se puede comparar. 2000-2009 es la única década completa. 2010-2016 es incompleta (7 años) porque el dataset termina en octubre de 2016.
- **Episodios sin guion:** 36 de los 600 episodios no tienen líneas de diálogo en el dataset. Se marcan en lugar de borrarlos de `episodios`, porque tienen rating y audiencia y sirven para análisis que no dependen del guion; se excluyen solo de la base de protagonismo. La base final queda con **564 episodios**.
- **Los nulos de rating (3 episodios) y la temporada 28 (4 episodios, incompleta) ya estaban entre esos 36**, por lo que no hubo que excluir episodios por esos motivos: la base de 564 ya no los contiene. Así, no se rellenó ningún nulo con el promedio, lo que habría inventado datos.
-  **Limitación:** de los 36 episodios sin guion, 33 pertenecen a 2010-2016, de modo que ese período pierde el 21,9% de sus episodios (118 de 151 quedan en la base). Las conclusiones de ese período se apoyan en menos episodios y deben leerse con cautela. El dataset no permite saber por qué falta el guion de esos episodios, y no se puede asegurar que la falta sea aleatoria.



In [17]:
# 1. Fecha, año y década
episodios["original_air_date"] = pd.to_datetime(episodios["original_air_date"], errors="coerce")
print("Fechas inválidas:", episodios["original_air_date"].isnull().sum())
episodios["anio"] = episodios["original_air_date"].dt.year

def asignar_decada(anio):
    if anio <= 1999:
        return "1989-1999"
    elif anio <= 2009:
        return "2000-2009"
    else:
        return "2010-2016"

episodios["decada"] = episodios["anio"].apply(asignar_decada)
print(episodios.groupby("decada").size())

# 2. Unión con el protagonismo y marca de episodios sin guion
analisis = episodios.merge(por_ep, left_on="id", right_on="episode_id", how="left")
print(len(episodios), len(analisis))   # deben ser iguales (600 y 600)
analisis["tiene_guion"] = analisis["n_palabras"].notna()
print(analisis["tiene_guion"].value_counts())   # 564 True / 36 False

# 3. ¿Qué porcentaje de cada período no tiene guion?
sin_guion = analisis[~analisis["tiene_guion"]]
total_por_decada = analisis.groupby("decada").size()
faltantes_por_decada = sin_guion.groupby("decada").size()
print(faltantes_por_decada)
print((faltantes_por_decada / total_por_decada * 100).round(1))

# 4. Base final y controles
base = analisis[analisis["tiene_guion"]].copy()
print("Episodios en la base final:", len(base))                 # 564
print("Episodios sin rating en la base:", base["imdb_rating"].isnull().sum())   # debe dar 0
print("Episodios de la temporada 28 en la base:", (base["season"] == 28).sum()) # debe dar 0
print(base.groupby("decada").size())

Fechas inválidas: 0
decada
1989-1999    235
2000-2009    214
2010-2016    151
dtype: int64
600 600
tiene_guion
True     564
False     36
Name: count, dtype: int64
decada
2000-2009     3
2010-2016    33
dtype: int64
decada
1989-1999     NaN
2000-2009     1.4
2010-2016    21.9
dtype: float64
Episodios en la base final: 564
Episodios sin rating en la base: 0
Episodios de la temporada 28 en la base: 0
decada
1989-1999    235
2000-2009    211
2010-2016    118
dtype: int64


### Asistencia de IA (Parte 1, etapa 2)
**Herramienta:** Claude

**Celdas con asistencia:** pregunta de investigación (3.2) y transformaciones 1 y 2 (3.3).

**Tipo de ayuda:** aportó ideas y código.

**Prompt utilizado**  « Quiero que te comportes como un analista de datos. El tp dejó una lista de transformaciones para esta etapa, pero quiero que agregues más cosas.» Se adjuntaron la lista de transformaciones, el notebook con la exploración inicial y las pautas del TP.

## 3.3 Resumen estadístico agrupado con `.groupby()`

**Idea general (en palabras simples):** `.groupby()` divide la tabla en "grupos" (por ejemplo, un grupo por década), calcula un número resumen en cada grupo (promedio, suma, cantidad) y permite comparar los grupos entre sí. Es como separar los episodios en tres pilas según su época y sacar el promedio de cada pila.

### Variables que usamos y por qué

| Variable | Tabla | Qué representa | Por qué la usamos |
|---|---|---|---|
| `character_id` | diálogos | Identificador único del personaje | Evita errores por nombres escritos distinto ("Bart", "Adult Bart") |
| `word_count` | diálogos | Palabras de cada línea | Mide cuánto "habla" un personaje (una línea larga pesa más que una corta) |
| `episode_id` | diálogos | Episodio al que pertenece la línea | Permite sumar el diálogo por episodio y unir con la tabla de episodios |
| `pct_principales` | construida | % de palabras del episodio dichas por Homer, Marge, Bart y Lisa | Medida de **protagonismo**, comparable entre episodios |
| `imdb_rating` | episodios | Nota promedio del episodio en IMDb (0–10) | Medida de **valoración** |
| `decada` / `season` | episodios | Período de emisión (3 grupos) / temporada | Permiten ver si la relación cambia con el tiempo |

**Variables que dejamos afuera:** `views`, `imdb_votes`, `us_viewers_in_millions`, `image_url`, `video_url`. No responden a la pregunta y `views`/audiencia tienen nulos. La audiencia podría servir en una extensión.

### Preparación: nombre de cada personaje

Agregamos una columna con el nombre "oficial" de cada `character_id` (el del ranking) y fijamos el orden de los cuatro principales, para que las tablas y gráficos salgan siempre en el mismo orden (de más a menos diálogo) y no alfabético.

In [18]:
# Nombre canónico de cada personaje (el del ranking) y orden de los 4 principales
dialogos_h["nombre"] = dialogos_h["character_id"].map(ranking["nombre"])
orden_principales = ranking.head(4)["nombre"].tolist()
print("Principales (de más a menos diálogo):", orden_principales)

# Orden cronológico de las décadas
orden_decadas = ["1989-1999", "2000-2009", "2010-2016"]

Principales (de más a menos diálogo): ['Homer Simpson', 'Marge Simpson', 'Bart Simpson', 'Lisa Simpson']


### Estadístico agrupado 1: ¿Quién concentra el diálogo? (por personaje y por década)

**Qué hacemos:** agrupamos las palabras habladas por década y por personaje, y calculamos qué porcentaje del total de palabras de cada década dice cada uno de los cuatro principales.

**Por qué:** responde la primera parte de la pregunta (*¿qué personajes concentran el diálogo?*) y muestra si el reparto cambió con el tiempo. Usamos **porcentajes** y no palabras absolutas porque las décadas tienen distinta cantidad de episodios.

In [19]:
# Unimos la década a cada línea de diálogo
dh = dialogos_h.merge(episodios[["id", "decada"]], left_on="episode_id", right_on="id",
                      how="left", suffixes=("", "_ep"))

# Total de palabras habladas por década (el "100%" de cada época)
palabras_decada = dh.groupby("decada")["word_count"].sum()

# Palabras de cada principal por década -> filas = década, columnas = personaje
pal_principales = (dh[dh["es_principal"]]
                   .groupby(["decada", "nombre"])["word_count"].sum()
                   .unstack("nombre")[orden_principales])

# Pasamos a porcentaje del total de la década
pct_decada = pal_principales.div(palabras_decada, axis=0) * 100
pct_decada["Total 4 principales"] = pct_decada.sum(axis=1)
pct_decada["Resto de personajes"] = 100 - pct_decada["Total 4 principales"]
pct_decada = pct_decada.loc[orden_decadas].round(1)
display(pct_decada)

# Para comparar: toda la serie
pct_serie = ranking.head(4)["pct_del_total"].round(1)
print("Toda la serie (% de las palabras):")
print(pct_serie.to_string(), "\nTotal 4 principales:", round(pct_serie.sum(), 1), "%")

nombre,Homer Simpson,Marge Simpson,Bart Simpson,Lisa Simpson,Total 4 principales,Resto de personajes
decada,,,,,,
1989-1999,21.2,9.3,9.1,7.3,46.9,53.1
2000-2009,21.2,10.4,7.8,7.9,47.3,52.7
2010-2016,19.9,9.0,8.5,8.2,45.6,54.4


Toda la serie (% de las palabras):
character_id
2    21.0
1     9.6
8     8.5
9     7.7 
Total 4 principales: 46.8 %


**Cómo leer este resultado:**
- Cada fila es una época y cada número es "de cada 100 palabras dichas en esa época, cuántas dice ese personaje".
- En toda la serie Homer dice cerca del 21% de las palabras y los cuatro principales juntos rondan el 47%: **casi la mitad del diálogo está en solo cuatro personajes**, de entre más de 6.000 nombres.
- Mirar *Total 4 principales*: si **baja** de una década a otra, el diálogo se repartió hacia personajes secundarios; si **sube**, la familia ganó protagonismo.
- **Resultado:** el reparto del diálogo es muy estable en el tiempo. Los cuatro principales dicen 46,9% de las palabras en 1989-1999, 47,3% en 2000-2009 y 45,6% en 2010-2016: casi la mitad en cada época. Hay cambios chicos dentro del grupo: Homer baja de 21,2% a 19,9%, Lisa sube de 7,3% a 8,2%,  Marge (9,3% → 10,4% → 9,0%) y Bart (9,1% → 7,8% → 8,5%) oscilan sin una tendencia clara.

### Estadístico agrupado 2: protagonismo y valoración por década y por temporada

**Qué hacemos:** agrupamos los 564 episodios por década (y luego por temporada) y calculamos cantidad de episodios, promedio, mediana y desvío del protagonismo (`pct_principales`) y del rating IMDb.

**Por qué:** muestra si, **en promedio**, protagonismo y valoración se movieron juntos en el tiempo. Se incluye la **mediana** (valor del medio) porque episodios extremos, como el de rating 4,5, distorsionan el promedio. La tabla por **temporada** es la que alimentará el gráfico de líneas.

In [20]:
agregados = dict(
    episodios=("id", "count"),
    protagonismo_prom=("pct_principales", "mean"),
    protagonismo_mediana=("pct_principales", "median"),
    protagonismo_desvio=("pct_principales", "std"),
    rating_prom=("imdb_rating", "mean"),
    rating_mediana=("imdb_rating", "median"),
    rating_desvio=("imdb_rating", "std"),
)

resumen_decada = base.groupby("decada").agg(**agregados).loc[orden_decadas].round(2)
display(resumen_decada)

# Misma idea, temporada por temporada (para el gráfico de líneas)
resumen_temporada = base.groupby("season").agg(**agregados).round(2)
display(resumen_temporada.head())

,episodios,protagonismo_prom,protagonismo_mediana,protagonismo_desvio,rating_prom,rating_mediana,rating_desvio
decada,,,,,,,
1989-1999,235,47.14,48.20,13.52,8.07,8.1,0.52
2000-2009,211,47.25,47.77,12.19,7.05,7.1,0.40
2010-2016,118,45.82,47.05,13.19,6.79,6.9,0.47


,episodios,protagonismo_prom,protagonismo_mediana,protagonismo_desvio,rating_prom,rating_mediana,rating_desvio
season,,,,,,,
1,13,55.90,56.80,8.52,7.81,7.80,0.28
2,22,51.24,52.47,11.47,8.04,8.05,0.34
3,24,44.82,44.75,12.65,8.15,8.20,0.33
4,22,44.70,44.37,12.66,8.27,8.20,0.45
5,22,43.34,40.28,10.60,8.34,8.40,0.38


**Cómo leer este resultado:**
- `protagonismo_prom`: porcentaje típico de palabras de un episodio dicho por Homer, Marge, Bart y Lisa.
- `rating_prom`: nota típica de un episodio. Si baja década tras década mientras el protagonismo se mantiene o sube, hay una señal de que **la caída de la nota no se explica por el peso de la familia** (otro factor, como el desgaste de la serie, pesaría más).
- `desvío`: cuánto varían los episodios dentro del grupo. Un desvío chico significa episodios muy parecidos entre sí.
- **Aviso:** 2010-2016 tiene solo 118 episodios por la falta de guiones; sus promedios son menos firmes.
- **Resultado:** el protagonismo casi no se movió (47,1% → 47,3% → 45,8% en promedio), pero el rating cayó fuerte: 8,07 → 7,05 → 6,79, es decir, 1,3 puntos menos entre la primera y la última década. Las medianas cuentan la misma historia (8,1; 7,1; 6,9).
- Por temporada se ve mejor: el rating sube hasta un máximo de cerca de 8,3 en las temporadas 5 a 7 y baja hasta 6,7 en la temporada 26, mientras que el protagonismo oscila entre 41% y 56% sin tendencia. La correlación entre número de temporada y rating es de -0,75; entre número de temporada y protagonismo, de -0,03.
- El protagonismo varía mucho de un episodio a otro (desvío de 12 a 14 puntos; va de 9% a 78%), mientras que el rating varía poco dentro de cada década (desvío de 0,4 a 0,5).
- **Lectura:** la caída de la valoración **no coincide** con ningún cambio en el peso de la familia, por lo que ese peso no la explica.

### Estadístico agrupado 3: valoración según nivel de protagonismo (cuartiles)

**Qué hacemos:** dividimos los episodios en 4 grupos de igual tamaño según su protagonismo (Q1 = episodios donde la familia habla **menos**; Q4 = donde habla **más**) y calculamos el rating promedio de cada grupo, para toda la serie y por década.

**Por qué:** es la forma más intuitiva de ver la asociación: si al pasar de Q1 a Q4 el rating sube o baja de manera ordenada, hay una relación. Mostramos también **cuántos episodios** hay en cada celda, porque un promedio de 10 episodios es mucho menos confiable que uno de 40.

In [21]:
etiquetas = ["Q1 (menos)", "Q2", "Q3", "Q4 (más)"]
base["nivel_protagonismo"] = pd.qcut(base["pct_principales"], q=4, labels=etiquetas)

print("Toda la serie:")
display(base.groupby("nivel_protagonismo", observed=True)["imdb_rating"]
        .agg(episodios="count", rating_prom="mean", rating_mediana="median").round(2))

# Rating promedio y cantidad de episodios, por década y nivel de protagonismo
cuartiles_decada = (base.groupby(["decada", "nivel_protagonismo"], observed=True)["imdb_rating"]
                    .mean().unstack("nivel_protagonismo").loc[orden_decadas].round(2))
n_cuartiles = (base.groupby(["decada", "nivel_protagonismo"], observed=True)["imdb_rating"]
               .count().unstack("nivel_protagonismo").loc[orden_decadas])
print("Rating promedio por década y nivel de protagonismo:")
display(cuartiles_decada)
print("Cantidad de episodios en cada celda:")
display(n_cuartiles)

Toda la serie:


,episodios,rating_prom,rating_mediana
nivel_protagonismo,,,
Q1 (menos),141,7.43,7.3
Q2,141,7.38,7.2
Q3,141,7.42,7.3
Q4 (más),141,7.46,7.3


Rating promedio por década y nivel de protagonismo:


nivel_protagonismo,Q1 (menos),Q2,Q3,Q4 (más)
decada,,,,
1989-1999,8.12,8.16,8.04,8.00
2000-2009,6.98,7.10,7.13,6.95
2010-2016,6.75,6.66,6.79,7.00


Cantidad de episodios en cada celda:


nivel_protagonismo,Q1 (menos),Q2,Q3,Q4 (más)
decada,,,,
1989-1999,62,50,56,67
2000-2009,47,59,55,50
2010-2016,32,32,30,24


**Cómo leer este resultado:**
- Recorrer cada fila de izquierda a derecha (Q1 → Q4). Si los números **suben**, a más protagonismo mejor nota; si **bajan**, al revés; si **no siguen un orden**, no hay relación clara.
- Diferencias de pocas décimas son chicas: el 50% central de los episodios está entre 6,9 y 8,0.
- Si el patrón se repite en las tres décadas, la asociación **se mantiene**; si cambia de signo, **no se mantiene**.
- Los cuartiles se calculan sobre toda la serie, así que cada década no tiene la misma cantidad de episodios por grupo: por eso se muestra la tabla de cantidades.
- **Resultado:** en toda la serie el rating promedio es prácticamente igual en los cuatro grupos (7,43; 7,38; 7,42; 7,46): pasar de "poco" a "mucho" protagonismo no cambia la nota.
- Por década el patrón **no se repite**: en 1989-1999 hay una leve baja (8,12 en Q1 → 8,00 en Q4), en 2000-2009 no hay orden (6,98; 7,10; 7,13; 6,95) y en 2010-2016 aparece una suba en el último grupo (6,75 en Q1 → 7,00 en Q4). Todas las diferencias son chicas (de 0,12 a 0,34 puntos entre el mejor y el peor grupo de cada década).
- La suba de 2010-2016 se apoya en pocos episodios (24 en Q4 y 32 en Q1), así que conviene contrastarla con la correlación del próximo estadístico.

### Estadístico complementario 4: correlación por década, con su margen de error

**Qué hacemos:** calculamos la **correlación** entre protagonismo y rating en cada década. Es un número entre -1 y 1: cerca de 0 significa "no hay relación"; positivo, "cuando uno sube el otro tiende a subir"; negativo, "cuando uno sube el otro tiende a bajar". Usamos Pearson (relación en línea recta) y Spearman (por ranking, menos sensible a casos extremos).

Agregamos dos columnas para no sobreinterpretar:
- **p-valor:** probabilidad de ver una correlación así de grande *si en realidad no hubiera ninguna relación*. Por convención, menos de 0,05 sugiere que no es solo azar.
- **Intervalo de confianza 95%:** rango donde probablemente está la correlación "verdadera". Si el rango incluye el 0, no podemos asegurar que haya relación.

**Por qué:** con pocos episodios (como en 2010-2016) una correlación chica puede ser puro azar; el intervalo permite verlo. **Correlación no es causalidad.**

In [22]:
from scipy import stats

def resumen_corr(df, x="pct_principales", y="imdb_rating"):
    n = len(df)
    r, p = stats.pearsonr(df[x], df[y])
    rho, p_s = stats.spearmanr(df[x], df[y])
    # Intervalo de confianza 95% para Pearson (transformación de Fisher)
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    return pd.Series({"episodios": n, "pearson": r, "p_valor": p,
                      "ic95_inf": np.tanh(z - 1.96 * se), "ic95_sup": np.tanh(z + 1.96 * se),
                      "spearman": rho, "p_valor_spearman": p_s})

corr_decada = base.groupby("decada")[["pct_principales", "imdb_rating"]].apply(resumen_corr).loc[orden_decadas]
corr_decada.loc["Toda la serie"] = resumen_corr(base)
corr_decada["episodios"] = corr_decada["episodios"].astype(int)
display(corr_decada.round(3))

,episodios,pearson,p_valor,ic95_inf,ic95_sup,spearman,p_valor_spearman
decada,,,,,,,
1989-1999,235,-0.125,0.056,-0.249,0.003,-0.123,0.059
2000-2009,211,0.003,0.963,-0.132,0.138,-0.035,0.609
2010-2016,118,0.236,0.010,0.058,0.400,0.230,0.012
Toda la serie,564,0.011,0.799,-0.072,0.093,0.016,0.704


**Cómo leer este resultado:**
- Regla práctica para el tamaño: entre -0,1 y 0,1 ≈ sin relación; hasta ±0,3 ≈ débil; ±0,3 a ±0,5 ≈ moderada; más de ±0,5 ≈ fuerte.
- Si Pearson y Spearman son parecidos, el resultado es robusto.
- Si el intervalo de confianza incluye el 0 (o el p-valor supera 0,05), **no hay evidencia suficiente** de relación en ese período.
- **Resultado en toda la serie:** no hay asociación. La correlación es de 0,01 (IC 95%: de -0,07 a 0,09; p = 0,80) con 564 episodios, y Spearman da lo mismo (0,02).
- **Por década, el signo cambia, así que la asociación no se mantiene:**
  - 1989-1999: -0,12 (IC 95%: de -0,25 a 0,00; p = 0,056). Es una relación débil y negativa, en el límite de lo que se puede distinguir del azar.
  - 2000-2009: 0,00 (p = 0,96). No hay relación.
  - 2010-2016: +0,24 (IC 95%: de 0,06 a 0,40; p = 0,010; Spearman 0,23). Es la única claramente distinta de cero y sigue siéndolo si se corrige por hacer tres comparaciones (umbral 0,05 / 3 ≈ 0,017). Es una relación débil a moderada y positiva.
- **Tamaño del efecto en 2010-2016:** el protagonismo explica solo alrededor del 5,6% de las diferencias de rating entre episodios. En números concretos, +10 puntos porcentuales de protagonismo se asocian con unos +0,08 puntos de rating (IC 95%: de +0,02 a +0,15). Es un efecto real pero pequeño.

### Estadístico complementario 5: ¿el paso del tiempo está distorsionando la asociación?

**El problema:** el rating de la serie cambia con los años y el protagonismo también puede cambiar. Si ambos se mueven con el tiempo, pueden parecer relacionados solo porque "viajan juntos en el calendario", aunque no tengan nada que ver entre sí.

**Qué hacemos:** a cada episodio le restamos el promedio de **su propia temporada** (tanto en protagonismo como en rating). Así cada episodio se compara solo con los de su misma temporada, y el efecto del tiempo queda eliminado. Luego recalculamos la correlación.

**Por qué:** es un control simple que fortalece la conclusión. Si la correlación **no cambia mucho**, la asociación (o su ausencia) no se debe al paso del tiempo.

In [23]:
# Desvío de cada episodio respecto del promedio de su temporada
base["protag_vs_temporada"] = base["pct_principales"] - base.groupby("season")["pct_principales"].transform("mean")
base["rating_vs_temporada"] = base["imdb_rating"] - base.groupby("season")["imdb_rating"].transform("mean")

corr_control = base.groupby("decada")[["protag_vs_temporada", "rating_vs_temporada"]].apply(
    lambda d: resumen_corr(d, "protag_vs_temporada", "rating_vs_temporada")).loc[orden_decadas]
corr_control.loc["Toda la serie"] = resumen_corr(base, "protag_vs_temporada", "rating_vs_temporada")
corr_control["episodios"] = corr_control["episodios"].astype(int)

print("Correlación DENTRO de cada temporada (sin efecto del tiempo):")
display(corr_control[["episodios", "pearson", "p_valor", "ic95_inf", "ic95_sup"]].round(3))
print("Comparación con la correlación sin control:")
display(pd.DataFrame({"sin_control": corr_decada["pearson"],
                      "con_control": corr_control["pearson"]}).round(3))

Correlación DENTRO de cada temporada (sin efecto del tiempo):


,episodios,pearson,p_valor,ic95_inf,ic95_sup
decada,,,,,
1989-1999,235,-0.016,0.812,-0.143,0.113
2000-2009,211,0.016,0.813,-0.119,0.151
2010-2016,118,0.242,0.008,0.064,0.405
Toda la serie,564,0.056,0.182,-0.026,0.138


Comparación con la correlación sin control:


,sin_control,con_control
decada,,
1989-1999,-0.125,-0.016
2000-2009,0.003,0.016
2010-2016,0.236,0.242
Toda la serie,0.011,0.056


**Cómo leer este resultado:**
- Si `sin_control` y `con_control` son parecidos, el paso del tiempo **no** explica el resultado.
- Si cambian mucho (por ejemplo, una correlación que desaparece al controlar), la asociación original era en gran parte un efecto del calendario.
- **Resultado:**
  - 1989-1999: la correlación pasa de -0,12 a -0,02 (p = 0,81). La relación negativa **desaparece**: era un efecto del paso del tiempo. En las primeras temporadas el protagonismo bajó (de 55,9% en la temporada 1 a 43,3% en la 5) mientras el rating subía (de 7,81 a 8,34), y eso generaba una correlación negativa aparente.
  - 2000-2009: sin cambios (de 0,00 a 0,02).
  - 2010-2016: se **mantiene** (de 0,24 a 0,24; p = 0,008). Dentro de una misma temporada, los episodios con más protagonismo de la familia tienden a tener algo mejor nota.
  - Toda la serie: de 0,01 a 0,06 (IC 95%: de -0,03 a 0,14; p = 0,18). Sigue sin haber asociación general.
- **Lectura:** el control por temporada no cambia la conclusión general. Solo elimina la asociación negativa de la primera década, que era aparente, y confirma que la de 2010-2016 es robusta.

### Estadístico complementario 6: ¿algún personaje en particular se relaciona con la valoración?

**Qué hacemos:** en lugar de sumar los cuatro principales, calculamos el porcentaje de palabras de **cada uno** en cada episodio y lo correlacionamos con el rating, por década.

**Por qué:** puede ocurrir que el grupo en conjunto no muestre relación, pero uno de ellos sí (por ejemplo, que los episodios muy centrados en Homer se valoren distinto que los centrados en Lisa). Si no miramos por separado, ese efecto queda escondido.

**Precaución:** acá hacemos 12 correlaciones (4 personajes × 3 décadas) más 4 de toda la serie. Cuando se hacen muchas comparaciones, alguna puede salir "llamativa" solo por azar. Hay que ser prudentes con resultados aislados.

In [24]:
from scipy import stats

# % de palabras de cada principal en cada episodio
pal_ep_pers = (dialogos_h[dialogos_h["es_principal"]]
               .groupby(["episode_id", "nombre"])["word_count"].sum()
               .unstack("nombre", fill_value=0)[orden_principales])
pct_ep_pers = pal_ep_pers.div(por_ep.set_index("episode_id")["n_palabras"], axis=0) * 100

tmp = base.set_index("id")[["decada", "season", "imdb_rating"]].join(pct_ep_pers)

# (a) Matriz simple: personaje x período (correlación con el rating)
corr_personajes = tmp.groupby("decada").apply(
    lambda d: d[orden_principales].corrwith(d["imdb_rating"])).loc[orden_decadas].T
corr_personajes["Toda la serie"] = tmp[orden_principales].corrwith(tmp["imdb_rating"])
print("(a) Correlación con el rating:")
display(corr_personajes.round(2))

# (b) Detalle con p-valor, y repetido comparando cada episodio solo con los de su temporada
filas = []
for nombre in orden_principales:
    for periodo in orden_decadas + ["Toda la serie"]:
        d = tmp if periodo == "Toda la serie" else tmp[tmp["decada"] == periodo]
        r, p = stats.pearsonr(d[nombre], d["imdb_rating"])
        x = d[nombre] - d.groupby("season")[nombre].transform("mean")
        y = d["imdb_rating"] - d.groupby("season")["imdb_rating"].transform("mean")
        r_c, p_c = stats.pearsonr(x, y)
        filas.append({"personaje": nombre, "periodo": periodo, "episodios": len(d),
                      "r": r, "p_valor": p, "r_control_temp": r_c, "p_valor_control": p_c})
detalle_personajes = pd.DataFrame(filas).set_index(["personaje", "periodo"])
print("(b) Detalle con p-valores:")
display(detalle_personajes.round(4))

umbral = 0.05 / 16   # corrección de Bonferroni por 16 comparaciones
print(f"Umbral con corrección por 16 comparaciones: p < {umbral:.4f}")


(a) Correlación con el rating:


/tmp/ipykernel_3291/3563355159.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  corr_personajes = tmp.groupby("decada").apply(


decada,1989-1999,2000-2009,2010-2016,Toda la serie
Homer Simpson,0.02,0.11,0.17,0.08
Marge Simpson,-0.23,-0.20,0.07,-0.12
Bart Simpson,-0.00,0.05,0.11,0.07
Lisa Simpson,-0.08,-0.03,0.05,-0.07


(b) Detalle con p-valores:


episodios       r  p_valor  r_control_temp  \
personaje     periodo                                                     
Homer Simpson 1989-1999            235  0.0157   0.8104          0.1874   
              2000-2009            211  0.1133   0.1009          0.0869   
              2010-2016            118  0.1729   0.0611          0.1863   
              Toda la serie        564  0.0759   0.0715          0.1569   
Marge Simpson 1989-1999            235 -0.2290   0.0004         -0.2356   
              2000-2009            211 -0.1976   0.0039         -0.1787   
              2010-2016            118  0.0742   0.4244          0.0890   
              Toda la serie        564 -0.1204   0.0042         -0.1431   
Bart Simpson  1989-1999            235 -0.0007   0.9913         -0.0392   
              2000-2009            211  0.0469   0.4978          0.0845   
              2010-2016            118  0.1095   0.2380          0.1005   
              Toda la serie        564  0.0742   0.0784          0.0296   
Lisa Simpson  1989-1999            235 -0.0832   0.2036         -0.0965   
              2000-2009            211 -0.0263   0.7043         -0.0185   
              2010-2016            118  0.0494   0.5956          0.0387   
              Toda la serie        564 -0.0734   0.0814         -0.0413   

                             p_valor_control  
personaje     periodo                         
Homer Simpson 1989-1999               0.0039  
              2000-2009               0.2088  
              2010-2016               0.0434  
              Toda la serie           0.0002  
Marge Simpson 1989-1999               0.0003  
              2000-2009               0.0093  
              2010-2016               0.3380  
              Toda la serie           0.0007  
Bart Simpson  1989-1999               0.5495  
              2000-2009               0.2217  
              2010-2016               0.2790  
              Toda la serie           0.4832  
Lisa Simpson  1989-1999               0.1403  
              2000-2009               0.7896  
              2010-2016               0.6771  
              Toda la serie           0.3273

Umbral con corrección por 16 comparaciones: p < 0.0031


**Cómo leer este resultado:**
- Cada número es la correlación entre "% de palabras de ese personaje en el episodio" y el rating. Usar la misma escala del estadístico 4.
- Buscar patrones que **se repitan** entre décadas más que un valor aislado.
- La tabla (b) agrega el p-valor de cada correlación y la repite comparando cada episodio solo con los de su temporada (`r_control_temp`). Como se hacen 16 comparaciones, usamos un umbral más exigente: p < 0,003 (0,05 / 16).
- **Resultado:**
  - **Marge** es la que más se destaca y su relación es **negativa**: -0,23 en 1989-1999 (p = 0,0004) y -0,20 en 2000-2009 (p = 0,004), y -0,12 en toda la serie. El valor de 1989-1999 supera la corrección por comparaciones múltiples; el de 2000-2009 queda en el límite. En 2010-2016 desaparece (+0,07; p = 0,42). Los episodios con más palabras de Marge tendieron a tener algo menos de nota en los primeros 20 años.
  - **Homer** muestra una relación **positiva y chica**: 0,08 en toda la serie (p = 0,07), que sube a 0,16 (p = 0,0002) al comparar dentro de cada temporada. Con ese control, el valor de toda la serie supera la corrección por comparaciones múltiples.
  - **Bart y Lisa** no muestran relación: ningún valor supera ±0,11 en valor absoluto y todos los p-valores son mayores a 0,07.
- **Por qué importa:** Homer y Marge se reparten el mismo tiempo de pantalla y sus efectos tienen signo opuesto. Al sumar a los cuatro (estadístico 4) esos efectos se compensan, lo que ayuda a explicar por qué la correlación del grupo en toda la serie es casi cero. Que dos personajes se asocien con la nota en sentidos opuestos es un patrón estadístico; no demuestra que Marge "baje" o Homer "suba" la nota.

> **Asistencia de IA:** Claude (Claude Project "tp labo"). Se usó para proponer los estadísticos agrupados, el control por temporada y redactar las explicaciones. **Prompt:** «Actuá como un analista de datos senior [...] identificá las variables más relevantes para la pregunta de investigación y calculá al menos dos estadísticos de resumen agrupados con .groupby(), con explicación para personas sin conocimiento previo.» Luego se pidió revisar y mejorar el resultado, y finalmente se ejecutó el notebook con los datos reales.